# CEVAE実験: 潜在交絡因子Zが1次元の場合 (元 CEVAE実験_v4.ipynb)

Zが1次元の人工データで、CEVAEがX-learnerよりITEを正確に推定できるかを検証する。
ついでに「CEVAEのrunが成功/崩壊の二山に割れる」現象を発見し、観測データだけで
崩壊を見分けられる信号(mse_diff, z_var, z_t_corr, z_y_corr)を探す。

結果は `results/1d_main_results.csv`, `results/1d_signal_diagnostics.csv` に保存する
(2次元実験との比較は `compare_1d_vs_2d.ipynb` で行う)。

計算量の注意: 元の設定(ハイパラ探索n_iter=150、10 data_seed、崩壊seedの深掘り10 model_seed)は
重い。まず動作を見るだけなら `N_ITER` 等を小さくするとよい。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from dgp import generate_toy_1d
# xgboost(X-learner側)を先に読み込む: 先にtorchを読み込むと、環境によっては
# XGBoostの学習時にクラッシュすることがあるため(OpenMPライブラリの競合)。
from xlearner_baseline import run_xlearner, refit_xlearner_with_fixed_params, predict_xlearner

import torch
from cevae_model import CEVAEConfig, build_cevae, train_cevae, compute_val_elbo, estimate_ite
from diagnostics import (
    z_corr_general, compute_y_mse_diag, z_info_signals,
    confusion_at_threshold, best_threshold,
)

## データ生成(data_seed=0)とDGPの健全性チェック

In [ ]:
DATA_SEED = 0
X, T, Y, Y0, Y1, tau, Z = generate_toy_1d(seed=DATA_SEED)
d = X.shape[1]
N = len(Y)

proxy_cols = [f"X{j+1}" for j in range(d)]
df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T
df_full["Y"] = Y
df_full["tau"] = tau

df_train, df_test = train_test_split(df_full, test_size=0.3, random_state=DATA_SEED, stratify=df_full["T"])

print(f"N={N}, d={d}")
print(f"真のATE           : {tau.mean():.3f}")
print(f"ナイーブATE       : {Y[T==1].mean()-Y[T==0].mean():.3f}")
print(f"corr(T, Z)        : {np.corrcoef(T, Z)[0,1]:.3f}")
Xd = np.column_stack([np.ones(N), X])
coef, _, _, _ = np.linalg.lstsq(Xd, Z, rcond=None)
r2_zx = 1 - np.sum((Z - Xd @ coef) ** 2) / np.sum((Z - Z.mean()) ** 2)
print(f"R^2(Z|X) 線形回帰 : {r2_zx:.3f}  (低いほどXからZを直接は復元しにくい=proxyとして難しい)")

## X-learnerベースライン(data_seed=0でハイパラ探索。後の比較で使い回す)

In [ ]:
N_ITER_XGB = 50  # 元の設定。試すだけなら小さくしてよい
CV_XGB = 5

print("X-learner ハイパラ探索中...")
xl_tau_pred, xl_models_fixed = run_xlearner(df_train, df_test, proxy_cols, n_iter=N_ITER_XGB, cv=CV_XGB, seed=0)
xl_pehe_s0 = np.sqrt(np.mean((df_test["tau"].values - xl_tau_pred) ** 2))
print(f"X-learner √PEHE (data_seed=0): {xl_pehe_s0:.3f}")

## CEVAEのハイパラ探索(z_dim=1固定。val_ELBOが一番良いものを探す)

In [ ]:
import random

def df_to_tensors(df):
    return (
        torch.tensor(df[proxy_cols].values, dtype=torch.float32),
        torch.tensor(df["T"].values, dtype=torch.float32),
        torch.tensor(df["Y"].values, dtype=torch.float32),
    )

df_inner, df_val = train_test_split(df_train, test_size=0.2, random_state=0, stratify=df_train["T"])
x_inner, t_inner, y_inner = df_to_tensors(df_inner)
x_val, t_val, y_val = df_to_tensors(df_val)

SEARCH_SPACE = {
    "hidden": [32, 64, 128],
    "beta_kl_max": [0.1, 0.3, 0.5, 1.0],
    "kl_anneal_epochs": [50, 100, 150],
    "epochs": [150, 300],
    "weight_decay": [0.0, 1e-4, 1e-3],
    "lr": [1e-3, 5e-4, 3e-4],
}
N_ITER_CEVAE = 150  # 元の設定。試すだけなら小さくしてよい
TOP_K = 10           # 上位何件を、後のseedごとのval_ELBO再選択候補にするか

rng_search = random.Random(0)
search_records = []
cfg_list = []  # search_recordsと同じ順番でcfgそのものも保持しておく(DataFrame経由だと型が壊れるため)

print(f"CEVAE ランダムサーチ中(z_dim=1固定, n_iter={N_ITER_CEVAE})...")
for i in range(N_ITER_CEVAE):
    cfg = CEVAEConfig(
        z_dim=1,
        hidden=rng_search.choice(SEARCH_SPACE["hidden"]),
        beta_kl_max=rng_search.choice(SEARCH_SPACE["beta_kl_max"]),
        kl_anneal_epochs=rng_search.choice(SEARCH_SPACE["kl_anneal_epochs"]),
        epochs=rng_search.choice(SEARCH_SPACE["epochs"]),
        weight_decay=rng_search.choice(SEARCH_SPACE["weight_decay"]),
        lr=rng_search.choice(SEARCH_SPACE["lr"]),
        seed=0,
    )
    nets = build_cevae(cfg, d)
    train_cevae(nets, cfg, x_inner, t_inner, y_inner)
    val_elbo = compute_val_elbo(nets, x_val, t_val, y_val)
    search_records.append({"iter": i, "val_elbo": val_elbo, **cfg.__dict__})
    cfg_list.append(cfg)

df_search = pd.DataFrame(search_records).sort_values("val_elbo")
print("上位5件:")
print(df_search.head(5).to_string(index=False))

top_cfgs = [cfg_list[i] for i in df_search.head(TOP_K)["iter"]]

## メイン実験: 10 data_seedでCEVAE vs X-learnerを比較

In [ ]:
N_DATA_SEEDS = 10
main_records = []

for ds in range(N_DATA_SEEDS):
    print(f"\ndata_seed={ds}", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_1d(seed=ds)
    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts
    df_s["Y"] = Ys
    df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3, random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values
    te_idx = df_te_s.index.tolist()
    tr_idx = df_tr_s.index.tolist()
    Z_te_s = Zs[te_idx]

    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2, random_state=ds, stratify=df_tr_s["T"])
    x_inner_s, t_inner_s, y_inner_s = df_to_tensors(df_inner_s)
    x_val_s, t_val_s, y_val_s = df_to_tensors(df_val_s)

    # X-learner: data_seed=0で見つけたハイパラのまま、このseedのデータで学習し直す
    models_s = refit_xlearner_with_fixed_params(xl_models_fixed, df_tr_s, proxy_cols)
    tau_xl = predict_xlearner(models_s, df_te_s, proxy_cols)
    xl_pehe_s = float(np.sqrt(np.mean((tau_true_s - tau_xl) ** 2)))

    # CEVAE: 上位TOP_K件をこのseedのinner/valで試し、val_ELBO最良のものを採用
    best_elbo, best_idx = float("inf"), 0
    for k, base_cfg in enumerate(top_cfgs):
        cfg_tmp = CEVAEConfig(**{**base_cfg.__dict__, "seed": ds})
        nets_tmp = build_cevae(cfg_tmp, d)
        train_cevae(nets_tmp, cfg_tmp, x_inner_s, t_inner_s, y_inner_s)
        elbo = compute_val_elbo(nets_tmp, x_val_s, t_val_s, y_val_s)
        if elbo < best_elbo:
            best_elbo, best_idx = elbo, k

    cfg_final = CEVAEConfig(**{**top_cfgs[best_idx].__dict__, "seed": ds})
    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    nets_s = build_cevae(cfg_final, d)
    train_cevae(nets_s, cfg_final, xt[tr_idx], tt[tr_idx], yt[tr_idx])
    tau_cv = estimate_ite(nets_s, xt[te_idx])
    cv_pehe_s = float(np.sqrt(np.mean((tau_true_s - tau_cv) ** 2)))
    zc_s = z_corr_general(nets_s, xt[te_idx], tt[te_idx], yt[te_idx], Z_te_s)

    main_records.append({
        "data_seed": ds, "xl_pehe": xl_pehe_s, "cevae_pehe": cv_pehe_s,
        "z_corr": zc_s, "best_cfg_idx": best_idx,
    })
    print(f"  XL={xl_pehe_s:.3f}  CEVAE={cv_pehe_s:.3f}  z_corr={zc_s:.3f}")

df_main = pd.DataFrame(main_records)
print("\n=== 集計(mean±std) ===")
print(df_main[["xl_pehe", "cevae_pehe", "z_corr"]].agg(["mean", "std"]).round(3))
n_win = int((df_main["cevae_pehe"] < df_main["xl_pehe"]).sum())
print(f"\nCEVAE勝率: {n_win}/{N_DATA_SEEDS}")

df_main.to_csv("results/1d_main_results.csv", index=False)
print("\n→ results/1d_main_results.csv に保存しました")

## 崩壊シードの深掘り診断

上の実験でz_corrが低かった(崩壊した)data_seedについて、model_seedだけ変えて複数回学習し直し、
「崩壊が初期値運かどうか」と「観測データだけで崩壊を検出できる信号」を調べる。

In [ ]:
Z_CORR_COLLAPSE_TH = 0.85
FAILED_SEEDS = df_main.loc[df_main["z_corr"] <= Z_CORR_COLLAPSE_TH, "data_seed"].tolist()
N_MODEL_SEEDS = 10  # 元の設定。試すだけなら小さくしてよい

print(f"崩壊判定(z_corr<={Z_CORR_COLLAPSE_TH})されたdata_seed: {FAILED_SEEDS}")

signal_records = []

for ds in FAILED_SEEDS:
    print(f"\n=== data_seed={ds} ===", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_1d(seed=ds)
    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts
    df_s["Y"] = Ys
    df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3, random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values
    te_idx = df_te_s.index.tolist()
    tr_idx = df_tr_s.index.tolist()
    Z_te_s = Zs[te_idx]

    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2, random_state=ds, stratify=df_tr_s["T"])
    x_val_s, t_val_s, y_val_s = df_to_tensors(df_val_s)

    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    x_te_s, t_te_s = xt[te_idx], tt[te_idx]
    y_te_s_np = df_te_s["Y"].values

    best_idx_for_ds = int(df_main.loc[df_main["data_seed"] == ds, "best_cfg_idx"].iloc[0])
    base_cfg = top_cfgs[best_idx_for_ds]

    for ms in range(N_MODEL_SEEDS):
        cfg_ms = CEVAEConfig(**{**base_cfg.__dict__, "seed": ms})
        nets_ms = build_cevae(cfg_ms, d)
        train_cevae(nets_ms, cfg_ms, xt[tr_idx], tt[tr_idx], yt[tr_idx])

        # 答え合わせ専用
        tau_pred_ms = estimate_ite(nets_ms, x_te_s)
        pehe_ms = float(np.sqrt(np.mean((tau_true_s - tau_pred_ms) ** 2)))
        zc_ms = z_corr_general(nets_ms, x_te_s, t_te_s, yt[te_idx], Z_te_s)

        # 選択信号候補(真のtau・Zを使わない)
        val_elbo_ms = compute_val_elbo(nets_ms, x_val_s, t_val_s, y_val_s)
        mse_q, mse_p, mse_diff = compute_y_mse_diag(nets_ms, x_te_s, t_te_s, y_te_s_np)
        z_var, z_t_corr, z_y_corr = z_info_signals(nets_ms, xt[tr_idx], tt[tr_idx], yt[tr_idx])

        signal_records.append({
            "data_seed": ds, "model_seed": ms, "val_elbo": val_elbo_ms,
            "sqrt_pehe": pehe_ms, "z_corr": zc_ms,
            "mse_q": mse_q, "mse_p": mse_p, "mse_diff": mse_diff,
            "z_var": z_var, "z_t_corr": z_t_corr, "z_y_corr": z_y_corr,
        })
        collapsed = zc_ms <= Z_CORR_COLLAPSE_TH
        print(f"  ms={ms}  PEHE={pehe_ms:.3f}  z_corr={zc_ms:.3f}  elbo={val_elbo_ms:.3f}  "
              f"mse_diff={mse_diff:+.4f}  z_var={z_var:.4f}  z_t={z_t_corr:.3f}  z_y={z_y_corr:.3f}  "
              f"{'崩壊' if collapsed else '成功'}")

df_signal = pd.DataFrame(signal_records)
df_signal["collapsed"] = df_signal["z_corr"] <= Z_CORR_COLLAPSE_TH
df_signal.to_csv("results/1d_signal_diagnostics.csv", index=False)
print(f"\n合計 {len(df_signal)} run中、崩壊 {int(df_signal['collapsed'].sum())} run")
print("→ results/1d_signal_diagnostics.csv に保存しました")

## 各信号がどれだけ崩壊を見分けられるか(混同行列)

In [ ]:
candidate_signals = ["val_elbo", "mse_diff", "z_var", "z_t_corr", "z_y_corr"]
comparison = pd.DataFrame([best_threshold(df_signal, c) for c in candidate_signals])
comparison = comparison.sort_values("errors").reset_index(drop=True)

print(f"(参考) 全{len(df_signal)}run中、真の崩壊run数 = {int(df_signal['collapsed'].sum())}")
print("各信号の『最良の閾値』での混同行列(errorsが小さいほど分離が良い):\n")
comparison